### Trasformation 

\begin{equation}
E: (a, e, M, w)  \rightarrow X: (x,y,v_x, v_y)
\end{equation}

\begin{equation}
    \mathbb{J}_{X o E} =
    \begin{pmatrix} 
    \partial_ax & \partial_ex & \partial_wx & \partial_Mx\\
    \partial_ay & \partial_ey & \partial_wy & \partial_My \\
    \partial_av_x & \partial_ev_x & \partial_wv_x & \partial_Mv_x \\
    \partial_av_y & \partial_ev_y & \partial_wv_y & \partial_Mv_y 
    \end{pmatrix}
\end{equation}

In [1]:
import numpy as np
from Utils import CanonicalUnits, OrbitalElements, JaccobianComponents, Kepler, GravitationalParameters, X2E, computeNumericalJacobian
import spiceypy as spy
import scipy.optimize as optimize
from tqdm import tqdm

In [2]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [3]:
def jacobian_XoE(elements: OrbitalElements, grav_params: CanonicalUnits) -> np.array:
    partial_a = JaccobianComponents.partial_a(elements, grav_params)
    a_x = partial_a[0]
    a_y = partial_a[1]
    a_vx = partial_a[3]
    a_vy = partial_a[4]

    partial_e = JaccobianComponents.partial_e(elements, grav_params)
    e_x = partial_e[0]
    e_y = partial_e[1]
    e_vx = partial_e[3]
    e_vy = partial_e[4]

    partial_w = JaccobianComponents.partial_w(elements, grav_params)
    w_x = partial_w[0]
    w_y = partial_w[1]
    w_vx = partial_w[3]
    w_vy = partial_w[4]

    partial_M = JaccobianComponents.partial_M(elements, grav_params)
    M_x = partial_M[0]
    M_y = partial_M[1]
    M_vx = partial_M[3]
    M_vy = partial_M[4]

    matrix = np.array([[a_x, e_x, w_x, M_x], 
                      [a_y, e_y, w_y, M_y], 
                      [a_vx, e_vx, w_vx, M_vx], 
                      [a_vy, e_vy, w_vy, M_vy]])
    
    return matrix

In [4]:
def jacobian_XoE_num(q: float, e: float, w: float, M: float, mu: float) -> np.array:
    E = [q, e, 0, 0, w, M]
    X = spy.conics(E+[0, mu], 0)
    dX=np.array([1e-3]*6)
    args=dict(mu=mu)
    E_num, JEoX=computeNumericalJacobian(X2E,X,dX,**args)

    partial_xq = JEoX[0,0]
    partial_xe = JEoX[1,0]
    partial_xw = JEoX[4,0]
    partial_xM = JEoX[5,0]

    partial_yq = JEoX[0,1]
    partial_ye = JEoX[1,1]
    partial_yw = JEoX[4,1]
    partial_yM = JEoX[5,1]

    partial_vxq = JEoX[0,3]
    partial_vxe = JEoX[1,3]
    partial_vxw = JEoX[4,3]
    partial_vxM = JEoX[5,3]

    partial_vyq = JEoX[0,4]
    partial_vye = JEoX[1,4]
    partial_vyw = JEoX[4,4]
    partial_vyM = JEoX[5,4]

    Matrix = np.array([[partial_xq, partial_yq, partial_vxq, partial_vyq], 
                        [partial_xe, partial_ye, partial_vxe, partial_vye], 
                        [partial_xw, partial_yw, partial_vxw, partial_vyw], 
                        [partial_xM, partial_yM, partial_vxM, partial_vyM]])
    return Matrix


In [5]:
def trasformation_aewE_to_xyvxvy(a: float, e: float, w: float, M: float) -> tuple[float, float, float, float]:
    Omega = 0
    i = 0
    mu = CanonicalUnits().mu
    q = a*(1-e)

    state_vec = spy.conics([q, e, i, w, Omega, M]+[0, mu], 0)
    x = state_vec[0]
    y = state_vec[1]
    vx = state_vec[3]
    vy = state_vec[4]

    return x, y, vx, vy

def trasformation_xyvxvy_to_aewE(x: float, y: float, vx: float, vy: float) -> tuple[float, float, float, float]:

    mu = CanonicalUnits().mu
    elements = spy.oscelt([x, y, 0, vx, vy, 0], et=0, mu=mu)
    q = elements[0]
    e = elements[1]
    w = elements[4]
    M = elements[5]
    a = q/(1-e)

    return a, e, w, M

In [6]:
N = int(1e5)

a_uniform = np.random.uniform(0, 2, N)
e_uniform = np.random.uniform(0, 1, N)
w_uniform = np.random.uniform(0, 2*np.pi, N)
M_uniform = np.random.uniform(0, 2*np.pi, N)
q_uniform = a_uniform*(1-e_uniform)

x_uniform, y_uniform, vx_uniform, vy_uniform = trasformation_aewE_to_xyvxvy(a_uniform[0], e_uniform[0], w_uniform[0], M_uniform[0])
x_uniform, y_uniform, vx_uniform, vy_uniform

(0.10494377135795264,
 0.5298957972542903,
 -8.280566027190353,
 1.6348829399222322)

In [7]:
a_inverse, e_inverse, w_inverse, M_inverse = trasformation_xyvxvy_to_aewE(x_uniform, y_uniform, vx_uniform, vy_uniform)
a_inverse, e_inverse, w_inverse, M_inverse 

(0.5267727474053503,
 0.025473058521511462,
 4.494415132803292,
 3.1652167115327523)

In [8]:
jacobian_XoE_num(q_uniform[1], e_uniform[1], w_uniform[1], M_uniform[1], mu)

array([[ 0.12816164, -0.07708124,  0.27462948,  0.044282  ],
       [-0.07021649,  0.01454244, -0.15488881, -0.015714  ],
       [ 0.29859459,  0.09036509, -0.10891113, -0.10995409],
       [-0.22634687, -0.20777927, -0.07630019,  0.7777417 ]])

In [9]:
xyvxvy = np.zeros((N, 4))

for el in tqdm(range(N)):
    a = a_uniform[el]
    e = e_uniform[el]
    w = w_uniform[el]
    M = M_uniform[el]

    x, y, vx, vy = trasformation_aewE_to_xyvxvy(a, e, w, M)
    xyvxvy[el] = np.array([x, y, vx, vy])

100%|██████████| 100000/100000 [00:01<00:00, 51060.54it/s]


In [10]:
xyvxvy

array([[ 1.04943771e-01,  5.29895797e-01, -8.28056603e+00,
         1.63488294e+00],
       [ 4.92902621e-01, -3.00732424e+00,  8.69048487e-01,
         1.40521442e+00],
       [-1.24348516e+00,  6.70039510e-01,  3.77929302e-01,
        -4.70348240e+00],
       ...,
       [-1.89236426e-01,  2.04983727e-01,  1.21138096e+00,
        -8.99806113e+00],
       [-1.94254479e-02, -1.15968814e-03,  1.71918932e+01,
        -2.97603850e+01],
       [-2.19369506e+00,  3.38792220e-01, -2.10973831e-01,
        -1.91728832e+00]])

In [15]:
def P_aewM() -> float:
    a_max = 2; a_min = 0
    e_max = 1; e_min = 0
    w_max = 2*np.pi; w_min = 0
    M_max = 2*np.pi; M_min = 0
    return 1/(a_max - a_min) * 1/(e_max - e_min) * 1/(w_max - w_min) * 1/(M_max - M_min)

def P_xyvxvy(x: float, y: float, vx: float, vy: float) -> float:
    a, e, w, M = trasformation_xyvxvy_to_aewE(x, y, vx, vy)
    q = a*(1-e)
    J = jacobian_XoE_num(q, e, w, M, mu)
    det = 1.0/np.linalg.det(J)
    P = P_aewM() * abs(det)
    return P

def P_xyvxvy_vectorized(x: np.array, y: np.array, vx: np.array, vy: np.array) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    vx = np.asarray(vx)
    vy = np.asarray(vy)

    # Prepare output array
    shape = np.broadcast(x, y, vx, vy).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()

    for idx in range(x_flat.size):
        a, e, w, M = trasformation_xyvxvy_to_aewE(x_flat[idx], y_flat[idx], vx_flat[idx], vy_flat[idx])
        q = a*(1-e)
        J = jacobian_XoE_num(q, e, w, M, mu)
        det = 1.0/np.linalg.det(J)
        P.flat[idx] = P_aewM() * abs(det)

    return P.reshape(shape)

def surface_integral_P_xyvxvy(center, widths, n_points=8):
    """
    Calculate the surface integral of P_xyvxvy in a hypercube centered at (x, y, vx, vy)
    with dimensions (dx, dy, dvx, dvy) using Gauss-Legendre quadrature.

    Parameters:
        center: tuple/list/array of (x, y, vx, vy) center
        widths: tuple/list/array of (dx, dy, dvx, dvy) side lengths
        n_points: number of quadrature points per dimension

    Returns:
        Integral (float)
    """
    from numpy.polynomial.legendre import leggauss

    x0, y0, vx0, vy0 = center
    dx, dy, dvx, dvy = widths

    # Get Gauss-Legendre points and weights for [-1, 1]
    pts, wts = leggauss(n_points)

    # Map points from [-1, 1] to [center-width/2, center+width/2] for each dimension
    x_pts = x0 + 0.5*dx*pts
    y_pts = y0 + 0.5*dy*pts
    vx_pts = vx0 + 0.5*dvx*pts
    vy_pts = vy0 + 0.5*dvy*pts

    # Create meshgrid of all quadrature points
    X, Y, VX, VY = np.meshgrid(x_pts, y_pts, vx_pts, vy_pts, indexing='ij')
    WX, WY, WVX, WVY = np.meshgrid(wts, wts, wts, wts, indexing='ij')

    # Flatten for vectorized evaluation
    Xf = X.ravel()
    Yf = Y.ravel()
    VXf = VX.ravel()
    VYf = VY.ravel()
    WF = (WX * WY * WVX * WVY).ravel()

    # Evaluate P at all points
    Pf = P_xyvxvy_vectorized(Xf, Yf, VXf, VYf)

    # Integral is sum(P * weight) * volume factor
    integral = np.sum(Pf * WF) * (0.5*dx) * (0.5*dy) * (0.5*dvx) * (0.5*dvy)
    return integral

In [22]:
# Example: Compare theoretical (integral) and numerical (count) number of objects in a phase-space volume

# Define center and widths of the phase-space hypercube
c_x = 1
c_y = 0
v_x = 0
v_y = (mu/1)**0.5

dxy = 0.5
dvxy = 5000 * (1/AU_m) * year

center = (c_x, c_y, v_x, v_y)
widths = (dxy, dxy, dvxy, dvxy)

# Theoretical: compute expected number of objects in the volume by integrating the distribution
N_theoretical = surface_integral_P_xyvxvy(center, widths, n_points=8)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical * N}")

# Numerical: count number of objects in the volume from the sample
objsx = (abs(xyvxvy[:,0] - c_x) <= dxy/2) 
objsy = (abs(xyvxvy[:,1] - c_y) <= dxy/2) 
objsvx = (abs(xyvxvy[:,2] - v_x) <= dvxy/2) 
objsvy = (abs(xyvxvy[:,3] - v_y) <= dvxy/2) 

objects = objsx & objsy & objsvx & objsvy
N_numeric = objects.sum()
print(f'Numerical (count) number of objects inside volume: {N_numeric}')

Theoretical (integral) number of objects in volume: 2078.0170132408703
Numerical (count) number of objects inside volume: 108


In [ ]:
def trasformation_aewE_to_xyvxvy(a: np.array, e: np.array, w: np.array, E: np.array):
    Omega = 0
    i = 0
    mu = CanonicalUnits().mu

    sqrt_e = (1-e)**0.5
    h = (mu * a * (1 - e**2))**0.5
    r = a * (1 - e * np.cos(E))
    term = mu * a / (h * r)

    A = np.cos(w) * np.cos(Omega) - np.cos(i) * np.sin(Omega) * np.sin(w)
    B = np.sin(w) * np.cos(Omega) + np.cos(i) * np.sin(Omega) * np.cos(w)
    C = np.sin(Omega) * np.cos(w) + np.cos(i) * np.cos(Omega) * np.sin(w)
    D = np.sin(Omega) * np.sin(w) - np.cos(i) * np.cos(Omega) * np.cos(w)

    x = (a * (np.cos(E) - e) * A - a * sqrt_e * np.sin(E) * B)
    y = (a * (np.cos(E) - e) * C - a * sqrt_e * np.sin(E) * D)
    vx = (-term * (np.cos(E) - e) * B - term * (1 - e**2)**0.5 * np.sin(E) * A - (mu * e / h) * B)
    vy = (-term * (np.cos(E) - e) * D - term * (1 - e**2)**0.5 * np.sin(E) * C - (mu * e / h) * D)

    xyvxvy = np.column_stack((x, y, vx, vy))
    return xyvxvy

def trasformation_xyvxvy_to_aewE(x: np.array, y: np.array, vx: np.array, vy: np.array):
    """
    Inverse transformation: Given (x, y, vx, vy), return a, e, w, E.
    Assumes Omega=0, i=0 as in the direct transformation.
    """
    mu = CanonicalUnits().mu
    # 1. Compute r and v
    x = np.asarray(x)
    y = np.asarray(y)
    vx = np.asarray(vx)
    vy = np.asarray(vy)
    r_vec = np.stack([x, y], axis=-1)
    v_vec = np.stack([vx, vy], axis=-1)
    r = np.linalg.norm(r_vec, axis=-1)
    v = np.linalg.norm(v_vec, axis=-1)

    # 2. Compute specific angular momentum (scalar, since 2D)
    h = x * vy - y * vx

    # 3. Compute eccentricity vector
    # e_vec = (v^2/mu - 1/r) * r_vec - (r·v)/mu * v_vec
    r_dot_v = x * vx + y * vy
    e_vec = ((v**2 - mu/r)/mu)[...,None] * r_vec - (r_dot_v/mu)[...,None] * v_vec
    e = np.linalg.norm(e_vec, axis=-1)

    # 4. Compute semi-major axis
    a = 1 / (2/r - v**2/mu)

    # 5. Compute argument of periapsis w
    # Perifocal x-axis is along e_vec, so w = atan2(e_y, e_x)
    w = np.arctan2(e_vec[...,1], e_vec[...,0])
    w = np.mod(w, 2*np.pi)

    # 6. Compute Eccentric Anomaly E
    # Rotate (x, y) into perifocal frame (x_p, y_p)
    # Perifocal x-axis is along e_vec
    # Build rotation matrix to align e_vec with x-axis
    # For each vector, angle = w, so rotate by -w
    cosw = np.cos(w)
    sinw = np.sin(w)
    x_p =  cosw * x + sinw * y
    y_p = -sinw * x + cosw * y

    # x_p = a (cosE - e)
    # y_p = a sqrt(1-e^2) sinE
    cosE = (x_p / a) + e
    sinE = y_p / (a * np.sqrt(1 - e**2))
    E = np.arctan2(sinE, cosE)
    E = np.mod(E, 2*np.pi)

    return a, e, w, E